# Professional Practice X3: Unsupervised Preprocessing Pipeline

Build a complete preprocessing pipeline using unsupervised methods.

**Goal:** Demonstrate preprocessing techniques:
- Feature scaling
- Outlier detection and removal
- Dimensionality reduction for feature engineering
- Feature clustering

**Use Case:** Prepare data for downstream tasks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans

# Generate synthetic dataset with outliers
X, y = make_classification(
    n_samples=1000, 
    n_features=20, 
    n_informative=15,
    n_redundant=5,
    random_state=42
)

# Add outliers
outlier_indices = np.random.choice(len(X), size=50, replace=False)
X[outlier_indices] += np.random.normal(0, 10, X[outlier_indices].shape)

print("Original data shape:", X.shape)
print(f"Features: {X.shape[1]}")

# Step 1: Outlier Detection
print("\n1. Detecting outliers with Isolation Forest...")
iso = IsolationForest(contamination=0.05, random_state=42)
outlier_labels = iso.fit_predict(X)
X_clean = X[outlier_labels == 1]
y_clean = y[outlier_labels == 1]

print(f"   Removed {np.sum(outlier_labels == -1)} outliers")
print(f"   Clean data shape: {X_clean.shape}")

# Step 2: Feature Scaling (Robust to outliers)
print("\n2. Scaling features with RobustScaler...")
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_clean)

# Step 3: Dimensionality Reduction
print("\n3. Reducing dimensions with PCA...")
pca = PCA(n_components=0.95, random_state=42)  # Keep 95% variance
X_pca = pca.fit_transform(X_scaled)

print(f"   Reduced from {X_scaled.shape[1]} to {X_pca.shape[1]} features")
print(f"   Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# Step 4: Feature Engineering - Add cluster assignments
print("\n4. Engineering features with K-Means clustering...")
kmeans = KMeans(n_clusters=3, random_state=42)
cluster_labels = kmeans.fit_predict(X_pca)

# Add cluster membership as one-hot encoded features
cluster_features = pd.get_dummies(cluster_labels, prefix='cluster').values
X_engineered = np.hstack([X_pca, cluster_features])

print(f"   Added {cluster_features.shape[1]} cluster features")
print(f"   Final feature count: {X_engineered.shape[1]}")

# Visualize preprocessing impact
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original data (first 2 features)
axes[0, 0].scatter(X[:, 0], X[:, 1], alpha=0.5, s=20)
axes[0, 0].set_title('1. Original Data (with outliers)', fontweight='bold')
axes[0, 0].set_xlabel('Feature 0')
axes[0, 0].set_ylabel('Feature 1')

# After outlier removal
axes[0, 1].scatter(X_clean[:, 0], X_clean[:, 1], alpha=0.5, s=20)
axes[0, 1].set_title('2. After Outlier Removal', fontweight='bold')
axes[0, 1].set_xlabel('Feature 0')
axes[0, 1].set_ylabel('Feature 1')

# After PCA
axes[1, 0].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=20)
axes[1, 0].set_title('3. After PCA Reduction', fontweight='bold')
axes[1, 0].set_xlabel('PC 1')
axes[1, 0].set_ylabel('PC 2')

# With cluster features
axes[1, 1].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, 
                  cmap='viridis', alpha=0.7, s=20)
axes[1, 1].set_title('4. Cluster-Based Features', fontweight='bold')
axes[1, 1].set_xlabel('PC 1')
axes[1, 1].set_ylabel('PC 2')

plt.tight_layout()
plt.show()

print("\n✅ Complete unsupervised preprocessing pipeline!")
print(f"\nPipeline Summary:")
print(f"  Input:  {X.shape[0]} samples, {X.shape[1]} features")
print(f"  Output: {X_engineered.shape[0]} samples, {X_engineered.shape[1]} features")